In [2]:
import pandas as pd



In [6]:
data = pd.read_csv('./resources/cnmf_factor_cluster_top_genes_200.csv', sep = ',')
data[0:5]

,Cluster 0 genes,Cluster 0 # of shared factors,Cluster 1 genes,Cluster 1 # of shared factors,Cluster 2 genes,Cluster 2 # of shared factors,Cluster 3 genes,Cluster 3 # of shared factors,Cluster 4 genes,Cluster 4 # of shared factors,...,Cluster 30 genes,Cluster 30 # of shared factors,Cluster 31 genes,Cluster 31 # of shared factors,Cluster 32 genes,Cluster 32 # of shared factors,Cluster 33 genes,Cluster 33 # of shared factors,Cluster 34 genes,Cluster 34 # of shared factors
0,FAM189A2,16,TNC,14,DPP6,12,CFI,3,ALCAM,13,...,ABCA1,2,ABLIM3,1,WDR62,34,HSPH1,6,ATP13A4,5
1,OGFRL1,14,CD44,14,CHST11,11,SNTG1,3,GALNT13,12,...,FNDC3B,2,NHS,1,C21ORF58,34,PPP1R15A,6,LINC00299,5
2,MAP3K5,14,VCL,13,MAP3K1,11,PLEKHG1,3,DNM3,12,...,PKP4,2,MXD1,1,MSH5,34,UBC,6,AQP4,5
3,ITPR2,14,SAMD4A,13,SLC24A3,11,PRKD1,3,ADGRL3,12,...,PLCB1,2,MYOF,1,SMC4,34,CCDC59,6,HIF3A,5
4,ETNPPL,14,IGFBP7,13,NRXN1,11,ETV6,3,MIR181A2HG,12,...,FAM20C,2,NAMPT,1,LMNB1,34,UBB,6,LINC01727,5


In [4]:
list(data['Cluster 0 genes'])


['FAM189A2',
 'OGFRL1',
 'MAP3K5',
 'ITPR2',
 'ETNPPL',
 'NRG3',
 'CD38',
 'FMN2',
 'LINC01088',
 'KCNN3',
 'DAAM2',
 'AC002429.2',
 'OBI1-AS1',
 'NTRK2',
 'SYTL4',
 'WDR49',
 'ADGRV1',
 'LIFR',
 'AQP4',
 'ID3',
 'OSBPL11',
 'DPP10',
 'SERPINI2',
 'TLR4',
 'NAA11',
 'MGAT4C',
 'AC026316.5',
 'EEPD1',
 'RASSF4',
 'AL392086.3',
 'SLC4A4',
 'EDNRB',
 'SLC39A11',
 'ATP1A2',
 'SLCO1C1',
 'AHCYL2',
 'SPON1',
 'SLC1A3',
 'GRAMD2B',
 'DTNA',
 'AC012405.1',
 'NKAIN3',
 'NTM',
 'SLC14A1',
 'DCLK2',
 'DCLK1',
 'ID4',
 'AC124854.1',
 'LINC01094',
 'PCDH9',
 'GABBR2',
 'PARD3B',
 'PDE8A',
 'LRIG1',
 'C5ORF64',
 'RNF19A',
 'SPARCL1',
 'AC093535.1',
 'FADS2',
 'PLEKHA5',
 'ASTN2',
 'ADAMTS9',
 'AC073941.1',
 'SLC24A4',
 'PAPPA',
 'AC068587.4',
 'FARP1',
 'SORL1',
 'ARHGAP26',
 'CADPS',
 'ST3GAL6',
 'ITPKB',
 'GABRB1',
 'FAM107A',
 'MIR99AHG',
 'ANK2',
 'AC107223.1',
 'PPP2R2B',
 'LPL',
 'AL589935.1',
 'MRVI1',
 'TNIK',
 'AL160272.1',
 'AC016766.1',
 'RANBP3L',
 'ARHGEF4',
 'ADCY2',
 'NPL',
 'KCNQ5',


In [ ]:
prompt = """
System prompt: You are an expert biologist summarising information about gene programs given gene lists as input.  
You output text that is both human readable AND that can easily be parsed into triples or quads (triple + reference),
i.e. when you provide information about gene programs that a gene or set of genes are involved in you structure sentences 
so that the relationships of genes to programs can be parsed ubambiguously.  e.g. avoid structuring sentences like this:

'Several genes in the list, including PCD9, CDt44, ITA6, COL45, COL5A, and SPARCL10, are classical mediators 
of cell–cell adhesion and ECM organization;"  In this case it is not possible to parse out which genes are asserted to be 
mediators of cell–cell adhesion and which are involved in ECM organization.' """

user_prompt_minus_gene_list = """

User Prompt: The following gene list represents a gene program found in some subset of malignant cells isolated 
from an IDH-mutant astrocytoma.  Please make predictions of how this gene program affects the  malignant cells 
that express it - including structure, function (biological processes), metabolic state, interactions with the 
ECM and other cells. To do this you should use evidence not only from the astrocytoma literature and other relevant cancer 
literature, but also from the normal development and function of astrocytes. 
Rank predictions more highly where multiple genes in the list are known to be involved in a relevant process. 
Where multiple genes are known to be required for that process, assess whether all required genes are present 
and rank higher if they are. """



In [ ]:
def gen_bib(bib):
    out = ['\n\n## References\n']
    index = 1
    for ref in bib: 
        #print(ref.values())
        out.append(f"- [{str(index)}] {' '.join(str(x) for x in ref.values())}")
        index += 1
    return '\n'.join(out)

In [ ]:
import requests
import concurrent.futures
import os
import time





def query_perplexity(o):
    """
    Runs perplexity API queries
    o = json file with two keys: 'plain query', 'contextual query'
    
    """

    # --- API Setup ---
    key = os.getenv("PERPLEXITY_API_KEY")
    url = "https://api.perplexity.ai/chat/completions"
    headers = {
    "accept": "application/json",
    "authorization": f"Bearer {key}",
    "content-type": "application/json"
    }
    
    base_payload = {
    "model": "sonar-deep-research",
    "return_citations": True,
        "search_domain_filter": [
            "pubmed.ncbi.nlm.nih.gov",
            "ncbi.nlm.nih.gov/pmc/",
            "sciencedirect.com",
            "nature.com",
            "cell.com",
            "frontiersin.org",
            "journals.plos.org",
            "wikipedia.org",
        ],
    "messages": [
        {"role": "system", "content": "You are an expert biologist. Your answers must be based on primary scientific literature and major reviews from peer-reviewed sources."},
        {"role": "user", "content": ""}
    ]
    }
    
    local_payload = base_payload.copy()
    
    # Plain query
    local_payload['messages'][1]['content'] = o['plain_query']
    try:
        plain_resp = requests.post(url, headers=headers, json=local_payload).json()
    except Exception as e:
        plain_resp = {"error": str(e)}

    # Add a 1-second delay between the two requests
    time.sleep(1) 


    # Contextual query
    local_payload['messages'][1]['content'] = o['contextual_query']
    try:
        contextual_resp = requests.post(url, headers=headers, json=local_payload).json()
    except Exception as e:
        contextual_resp = {"error": str(e)}
    
    # Store in the object
    o['plain_response'] = plain_resp
    o['contextual_response'] = contextual_resp

    return o

In [ ]:
# Use a ThreadPoolExecutor to run the query function on all items in the 'out' list
# This will run multiple requests at the same time, making the process much faster.

with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
    # The map function applies 'query_perplexity' to each item in 'out'
    # and returns the results in the same order.
    print("Starting API calls...")
    results = list(executor.map(query_perplexity, out))
    print("All API calls completed.")